In [0]:
%run ../gold/00_gold_helpers

In [0]:
logger = get_logger("gold_product_sales")

try:

    logger.info("Starting Gold Product Sales transformation")

    # ---------------------------------------------------------
    # Read Products
    # ---------------------------------------------------------

    logger.info("Reading Silver Products table")

    df1 = read_table("products_clean")

    logger.info("Silver Products table read successfully")

    logger.info(
        "Removing technical columns from Products"
    )

    df1 = df1.drop(
        "_ingestion_timestamp",
        "_source_file"
    )

    logger.info("Product technical columns removed")

    display(df1)

    # ---------------------------------------------------------
    # Read Sales
    # ---------------------------------------------------------

    logger.info("Reading Silver Sales table")

    df2 = read_table("sales_clean")

    logger.info("Silver Sales table read successfully")

    logger.info(
        "Removing technical columns from Sales"
    )

    df2 = df2.drop(
        "_ingestion_timestamp",
        "_source_file"
    )

    logger.info("Sales technical columns removed")

    logger.info('printing schema and data')
    df.printSchema()
    display(df)

    # ---------------------------------------------------------
    # Join Products and Sales
    # ---------------------------------------------------------

    logger.info(
        "Joining Sales with Products using broadcast join"
    )

    df = df2.join(
        broadcast(df1),
        "product_id",
        how="right"
    )

    logger.info(
        "Sales and Products broadcast join completed"
    )

    df.show()

    # ---------------------------------------------------------
    # Product Sales Aggregation
    # ---------------------------------------------------------

    logger.info(
        "Aggregating sales by product_id, product_name and category"
    )

    df_agg = (
        df
        .groupBy(
            "product_id",
            "product_name",
            "category"
        )
        .agg(
            sum(
                col("quantity")
            ).alias("units_sold"),

            sum(
                col("quantity") * col("unit_price")
            ).alias("revenue")
        )
    )

    logger.info(
        "Product sales aggregation completed"
    )

    # ---------------------------------------------------------
    # Create Schema
    # ---------------------------------------------------------

    logger.info(
        f"Creating schema if it does not exist: "
        f"{catalog_name}.{schema_name}"
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog_name}.{schema_name}
        """
    )

    logger.info(
        f"Schema ready: {catalog_name}.{schema_name}"
    )

    # ---------------------------------------------------------
    # Save Gold Table
    # ---------------------------------------------------------

    logger.info(
        "Saving Gold Product Sales Summary table"
    )

    save_table(
        df_agg,
        "products_sales_summary"
    )

    logger.info(
        "Gold Product Sales Summary table saved successfully"
    )

    logger.info(
        "Gold Product Sales transformation completed successfully"
    )

except Exception:

    logger.exception(
        "Gold Product Sales transformation failed"
    )

    raise